# Convolutional Classifier – ISIC 2020 Training Images

Trains a binary convolutional classifier (benign vs. malignant) on the JPEG images in `dataset/train`.

**Key design decisions**
| Parameter | Where handled | Rationale |
|---|---|---|
| Image resize (H × W) | `ISICDataset` constructor | Images stay on disk; we only resize when a sample is fetched. |
| Duplicate removal | `ISICDataset` constructor | Pass `duplicates_csv` to drop redundant images automatically. |
| Labels | `ISICDataset` constructor | Pass `labels_csv`; `__getitem__` returns `(tensor, label)` tuples. |
| Train/Val split | `random_split` (seed 42) | 80 % train, 20 % val; reproducible. |
| Class imbalance | `WeightedRandomSampler` (train only) | The ISIC dataset is heavily imbalanced (~98 % benign); over-sample the minority class during training only. |

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import sys

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, random_split
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from dataset import ISICDataset

print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

In [ ]:
# ── Parameters (edit here) ────────────────────────────────────────────────────
DATASET_DIR      = "../dataset/train"                                 # path relative to this notebook
LABELS_CSV       = "../dataset/ISIC_2020_Training_GroundTruth.csv"
DUPLICATES_CSV   = "../dataset/ISIC_2020_Training_Duplicates.csv"
IMAGE_HEIGHT     = 128          # target height after resize
IMAGE_WIDTH      = 128          # target width  after resize
BATCH_SIZE       = 16
NUM_WORKERS      = 4
VAL_SPLIT        = 0.2           # fraction of data held out for validation
BASE_CHANNELS    = 32           # number of filters in the first conv layer
DROPOUT          = 0.5          # dropout probability before the final FC layer
LEARNING_RATE    = 2 * 1e-4
NUM_EPOCHS       = 20
CHECKPOINT_PATH  = "classifier_isic.pth"
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")
print(f"Target image size: {IMAGE_HEIGHT} x {IMAGE_WIDTH}")

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
class ConvClassifier(nn.Module):
    """
    Lightweight convolutional binary classifier.

    Parameters
    ----------
    image_height  : int
    image_width   : int
    in_channels   : int   (3 for RGB)
    base_channels : int   Number of filters in the first conv layer.
                          Doubles with each down-sampling stage.
    dropout       : float Dropout probability before the final linear layer.
    num_classes   : int   Number of output classes (2 for benign/malignant).
    """

    def __init__(
        self,
        image_height:  int   = 128,
        image_width:   int   = 128,
        in_channels:   int   = 3,
        base_channels: int   = 32,
        dropout:       float = 0.5,
        num_classes:   int   = 2,
    ):
        super().__init__()

        c1, c2, c3, c4 = (
            base_channels,
            base_channels * 2,
            base_channels * 4,
            base_channels * 8,
        )

        # ── Feature extractor ────────────────────────────────────────────────
        # Four stride-2 conv blocks; each halves H and W.
        self.features = nn.Sequential(
            # Block 1: (N, in_channels, H,    W)    → (N, c1, H/2,  W/2)
            nn.Conv2d(in_channels, c1, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(c1),
            nn.ReLU(inplace=True),

            # Block 2: (N, c1, H/2,  W/2)  → (N, c2, H/4,  W/4)
            nn.Conv2d(c1, c2, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(c2),
            nn.ReLU(inplace=True),

            # Block 3: (N, c2, H/4,  W/4)  → (N, c3, H/8,  W/8)
            nn.Conv2d(c2, c3, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(c3),
            nn.ReLU(inplace=True),

            # Block 4: (N, c3, H/8,  W/8)  → (N, c4, H/16, W/16)
            nn.Conv2d(c3, c4, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(c4),
            nn.ReLU(inplace=True),
        )

        # Global average pooling → (N, c4)
        self.pool = nn.AdaptiveAvgPool2d(1)

        # ── Classifier head ───────────────────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(c4, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

In [ ]:
# ── Dataset & DataLoaders ─────────────────────────────────────────────────────
dataset = ISICDataset(
    root_dir       = DATASET_DIR,
    target_height  = IMAGE_HEIGHT,
    target_width   = IMAGE_WIDTH,
    labels_csv     = LABELS_CSV,
    duplicates_csv = DUPLICATES_CSV,
)

print(f"Total samples : {len(dataset)}")

# ── Collect all labels (no image I/O) ────────────────────────────────────────
all_labels = [
    dataset.label_map[os.path.splitext(os.path.basename(p))[0]]
    for p in dataset.image_paths
]
class_counts = np.bincount(all_labels)           # [n_benign, n_malignant]
print(f"Class counts  : benign={class_counts[0]:,}  malignant={class_counts[1]:,}")

# ── Train / Val split ─────────────────────────────────────────────────────────
n_total = len(dataset)
n_val   = int(n_total * VAL_SPLIT)
n_train = n_total - n_val

# Fixed seed for reproducibility
generator = torch.Generator().manual_seed(42)
train_subset, val_subset = random_split(dataset, [n_train, n_val], generator=generator)

print(f"Train samples : {len(train_subset):,}")
print(f"Val   samples : {len(val_subset):,}")

# ── WeightedRandomSampler on the train subset only ───────────────────────────
train_labels   = [all_labels[i] for i in train_subset.indices]
class_weights  = 1.0 / class_counts              # inverse-frequency weights
sample_weights = [class_weights[lbl] for lbl in train_labels]
train_sampler  = WeightedRandomSampler(
    weights     = torch.DoubleTensor(sample_weights),
    num_samples = len(train_subset),
    replacement = True,
)

train_loader = DataLoader(
    train_subset,
    batch_size  = BATCH_SIZE,
    sampler     = train_sampler,    # mutually exclusive with shuffle=True
    num_workers = NUM_WORKERS,
    pin_memory  = DEVICE == "cuda",
)

val_loader = DataLoader(
    val_subset,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = DEVICE == "cuda",
)

print(f"Train batches/epoch : {len(train_loader)}")
print(f"Val   batches/epoch : {len(val_loader)}")

In [ ]:
# ── Visualise a batch (sanity check) ─────────────────────────────────────────
sample_imgs, sample_labels = next(iter(train_loader))
print("Batch shape :", sample_imgs.shape)
print("Labels      :", sample_labels[:8].tolist())

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for i, ax in enumerate(axes):
    img = sample_imgs[i].permute(1, 2, 0).numpy()  # C,H,W → H,W,C
    img = (img * 0.5) + 0.5                         # un-normalise to [0, 1]
    img = img.clip(0, 1)
    ax.imshow(img)
    ax.set_title("malignant" if sample_labels[i].item() == 1 else "benign", fontsize=9)
    ax.axis("off")
plt.suptitle("Sample images from train DataLoader", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Model / optimiser / loss ──────────────────────────────────────────────────
model = ConvClassifier(
    image_height  = IMAGE_HEIGHT,
    image_width   = IMAGE_WIDTH,
    in_channels   = 3,
    base_channels = BASE_CHANNELS,
    dropout       = DROPOUT,
    num_classes   = 2,
).to(DEVICE)

print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
from tqdm import tqdm

train_losses     = []
train_accuracies = []
val_losses       = []
val_accuracies   = []

for epoch in range(1, NUM_EPOCHS + 1):

    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0

    pbar = tqdm(train_loader, desc=f"Epoch [{epoch:>3}/{NUM_EPOCHS}] train", leave=False)
    for imgs, labels in pbar:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        logits = model(imgs)
        loss   = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds         = logits.argmax(dim=1)
        correct      += (preds == labels).sum().item()
        total        += imgs.size(0)
        pbar.set_postfix(loss=f"{running_loss / total:.4f}", acc=f"{100.0 * correct / total:.1f}%")

    train_loss = running_loss / total
    train_acc  = 100.0 * correct / total
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    # ── Validate ─────────────────────────────────────────────────────────────
    model.eval()
    val_running_loss = 0.0
    val_correct      = 0
    val_total        = 0

    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc=f"Epoch [{epoch:>3}/{NUM_EPOCHS}] val  ", leave=False):
            imgs   = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            val_running_loss += loss.item() * imgs.size(0)
            preds             = logits.argmax(dim=1)
            val_correct      += (preds == labels).sum().item()
            val_total        += imgs.size(0)

    val_loss = val_running_loss / val_total
    val_acc  = 100.0 * val_correct / val_total
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    print(
        f"Epoch [{epoch:>3}/{NUM_EPOCHS}]  "
        f"train_loss={train_loss:.4f}  train_acc={train_acc:.1f}%  "
        f"val_loss={val_loss:.4f}  val_acc={val_acc:.1f}%"
    )

print("\nTraining complete.")

In [ ]:
# ── Training curves (Train vs Val) ────────────────────────────────────────────
epochs = range(1, NUM_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, train_losses,     marker="o", linewidth=1.5, label="Train")
ax1.plot(epochs, val_losses,       marker="s", linewidth=1.5, label="Val",  linestyle="--")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross-Entropy Loss")
ax1.set_title("Loss")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.6)

ax2.plot(epochs, train_accuracies,  marker="o", linewidth=1.5, label="Train")
ax2.plot(epochs, val_accuracies,    marker="s", linewidth=1.5, label="Val",  linestyle="--")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Accuracy")
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.6)

plt.suptitle("Classifier Training Curves (Train vs Val)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Save checkpoint ───────────────────────────────────────────────────────────
torch.save({
    "epoch"       : NUM_EPOCHS,
    "model_state" : model.state_dict(),
    "optim_state" : optimizer.state_dict(),
    "config": {
        "image_height"  : IMAGE_HEIGHT,
        "image_width"   : IMAGE_WIDTH,
        "base_channels" : BASE_CHANNELS,
        "dropout"       : DROPOUT,
        "num_classes"   : 2,
        "val_split"     : VAL_SPLIT,
    },
    "train_losses"     : train_losses,
    "train_accuracies" : train_accuracies,
    "val_losses"       : val_losses,
    "val_accuracies"   : val_accuracies,
}, CHECKPOINT_PATH)

print(f"Checkpoint saved to '{CHECKPOINT_PATH}'")